# 🌿 Leva-TTS — Quick Start

Generate your first Levantine-Arabic / English code-switching audio in a few cells.

**What this notebook does**
1. Installs `leva-tts` (with the maintained `coqui-tts` engine)
2. Loads the model (auto-downloads the fine-tuned checkpoint from HuggingFace)
3. Synthesizes speech with a built-in speaker, a code-switch sentence, zero-shot cloning, and streaming

👉 First set **Runtime ▸ Change runtime type ▸ T4 GPU**.

## Install Leva-TTS

> **Note on the engine:** the original Coqui `TTS` package is unmaintained and
> caps at Python < 3.12, so it won't install on Colab's current runtime.
> We install the actively-maintained **`coqui-tts`** fork instead — it exposes
> the **same `TTS` modules** (XTTS-v2) and supports Python 3.12, so it's a
> drop-in for Leva-TTS. **No kernel restart needed.**

This cell installs everything with the **current kernel's** interpreter
(`sys.executable`), so versions can't get crossed. First run takes ~3–5 min.

In [ ]:
import sys
print('Kernel Python:', sys.version.split()[0])   # Colab is 3.11/3.12 — that's fine here

# 1) Maintained Coqui fork (provides the `TTS` / XTTS modules on Python 3.12)
!{sys.executable} -m pip install -q 'coqui-tts==0.24.3'   # pinned: pairs with a transformers that XTTS needs

# 2) Leva-TTS itself. --no-deps so pip does NOT pull the old, incompatible `TTS`;
#    coqui-tts (above) already provides the engine + transformers.
!{sys.executable} -m pip install -q --no-deps "leva-tts"
#    ...then its small pure-Python deps that Colab may not ship (coqui-tts covers the rest)
!{sys.executable} -m pip install -q soundfile librosa huggingface_hub rich pandas sympy

# 3) System audio libraries
!apt-get -qq install -y espeak-ng ffmpeg > /dev/null

# 4) Sanity: the package imports and the XTTS engine module is available
import importlib
importlib.invalidate_caches()
import leva_tts, TTS
from TTS.tts.models.xtts import Xtts          # engine entry point
print('✅ leva_tts', leva_tts.__version__, '· TTS (coqui-tts)', TTS.__version__)

## Check the GPU

In [ ]:
# Verify a GPU runtime is active (Runtime ▸ Change runtime type ▸ T4 GPU)
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or "⚠️ No GPU — switch the runtime to a GPU!")

## Load the model

First run downloads the checkpoint + 10 reference speakers (~2 GB) — give it a minute.

In [ ]:
from leva_tts import LevaTTS, SPEAKERS

tts = LevaTTS(device="cuda", preprocess_text=True, verbose=False)
print("Available speakers:", SPEAKERS)

## Synthesize with a built-in speaker

`synthesize(text, speaker, **gen_params)` → `(wav, sr)`. `speaker` must be one of `SPEAKERS`.

In [ ]:
from IPython.display import Audio, display
import soundfile as sf

wav, sr = tts.synthesize(
    "هَلَّق أنا عم أشتغل على the new project اللي حكيتلك عنه.",
    speaker="Badr",
    temperature=0.65,
)
sf.write("badr.wav", wav, sr)
print(f"{len(wav)/sr:.2f}s @ {sr} Hz")
display(Audio(wav, rate=sr))

## Try other speakers & a pure-Levantine line

In [ ]:
wav, sr = tts.synthesize("كيفك اليوم؟ إنت شو عم تعمل هَلَّق؟", speaker="Amina")
display(Audio(wav, rate=sr))

## Zero-shot voice cloning

Upload your own 3–10 s reference clip and clone the voice. Run the cell, pick a `.wav`/`.mp3`.

In [ ]:
from google.colab import files
up = files.upload()                 # choose a short reference clip
ref_path = next(iter(up))

wav, sr = tts.zero_shot_synthesize("مرحبا، هَلَّق عم نجرب الـ voice cloning.", ref_path)
display(Audio(wav, rate=sr))

## Streaming

`stream(...)` yields audio chunks as they're generated (useful for low-latency playback).

In [ ]:
import numpy as np
chunks = []
for ch in tts.stream("بِدِّي أحكيلك عن the new feature هَلَّق، رح تعجبك كتير.", speaker="Mona"):
    chunks.append(ch)
print(f"received {len(chunks)} chunks")
display(Audio(np.concatenate(chunks), rate=24000))

## 🎉 Done!

**Generation params** (optional on every method): `temperature`, `length_penalty`,
`repetition_penalty`, `top_k`, `top_p`, `speed`.

Next: [inference server](02_inference_server.ipynb) ·
[evaluation](03_evaluation.ipynb) · [Gradio app](04_gradio_app.ipynb).